# ZERAVIA | Data Science Mini Project
## Customer Behaviour Analysis & Purchase Prediction
**Student:** Shreya Sharma  
**Program:** Data Science Training & Internship — First Month  
**Project type:** Beginner-level end-to-end classification workflow

> **Dataset note:** This notebook creates a reproducible *synthetic practice dataset* because no client dataset was supplied. The patterns and model metrics are therefore demonstrations, not claims about a real business. Replace the data-generation section with the client's authorised CSV before presenting results as real findings.

### Workflow
Business problem → data loading/understanding → Python fundamentals → cleaning → transformation → NumPy/Pandas → EDA and visualisation → statistics/outliers → feature engineering → train/test split → Logistic Regression → evaluation → recommendations.

## 1. Business problem definition
A retailer wants to understand customer purchasing behaviour and estimate whether a customer will make a purchase based on historical demographic, transaction and engagement indicators.

**Target:** `purchased` (1 = purchase, 0 = no purchase).  
**Expected output:** a predicted purchase probability and a class prediction for each customer.  
**Business value:** support campaign prioritisation and customer-segment analysis.

**Assumptions:** records represent individual customer-period observations; all fields are available before the predicted purchase; the generated data is synthetic; association does not imply causation. The target is suitable for supervised binary classification.

In [ ]:
# 2. Imports and reproducibility
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, RocCurveDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", None)
sns.set_theme()


## 3. Data loading and understanding
The following creates a realistic-shaped practice dataset. To use a supplied client CSV, replace the generation block with `df = pd.read_csv('customer_data.csv')`.

In [ ]:
# Synthetic customer dataset (practice only)
n = 1200
age = np.random.randint(18, 71, n)
income = np.random.normal(55000, 18000, n).clip(12000, 140000).round(0)
website_visits = np.random.poisson(5, n).clip(0, 25)
email_clicks = np.random.poisson(2, n).clip(0, 12)
previous_orders = np.random.poisson(3, n).clip(0, 20)
avg_order_value = np.random.normal(1800, 650, n).clip(150, 6500).round(2)
days_since_last_order = np.random.randint(0, 181, n)
membership = np.random.choice(["Standard", "Silver", "Gold"], n, p=[0.55, 0.30, 0.15])
region = np.random.choice(["North", "South", "East", "West"], n)

logit = (-1.1 + 0.035*website_visits + 0.12*email_clicks
         + 0.10*previous_orders + 0.000012*(income-50000)
         + 0.45*(membership=="Gold") + 0.18*(membership=="Silver")
         - 0.012*days_since_last_order)
prob = 1/(1+np.exp(-logit))
purchased = np.random.binomial(1, prob)

df = pd.DataFrame({
    "customer_id": np.arange(10001, 10001+n),
    "age": age, "annual_income": income, "website_visits": website_visits,
    "email_clicks": email_clicks, "previous_orders": previous_orders,
    "avg_order_value": avg_order_value, "days_since_last_order": days_since_last_order,
    "membership": membership, "region": region, "purchased": purchased
})

# Add a small amount of realistic data-quality issues for cleaning practice
for col in ["annual_income", "avg_order_value", "membership"]:
    idx = np.random.choice(df.index, size=12, replace=False)
    df.loc[idx, col] = np.nan
df = pd.concat([df, df.iloc[[5, 25]]], ignore_index=True)  # deliberate duplicate rows

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.rename("data_type").to_frame())
display(df.isna().sum().rename("missing_values").to_frame())
display(df.describe(include="all").T)


## 4. Python fundamentals
This section demonstrates common data types, lists, tuples, sets, dictionaries, conditionals and loops used for simple validation.

In [ ]:
numeric_columns = ["age", "annual_income", "website_visits", "email_clicks",
                   "previous_orders", "avg_order_value", "days_since_last_order"]
required_columns = ("customer_id", "purchased")  # tuple
unique_regions = set(df["region"].dropna().tolist())
column_types = {column: str(dtype) for column, dtype in df.dtypes.items()}

for col in numeric_columns:
    if col in df.columns:
        print(f"{col}: minimum={df[col].min()}, maximum={df[col].max()}")

if not set(df["purchased"].dropna().unique()).issubset({0, 1}):
    print("Warning: target contains values other than 0 and 1")
else:
    print("Target validation passed.")
print("Regions:", unique_regions)
print("Column types:", column_types)


## 5. Data cleaning
Remove exact duplicate records, standardise category labels, and impute missing values later inside the modelling pipeline. No customer is removed merely because a value is missing.

In [ ]:
before = len(df)
df = df.drop_duplicates().copy()
print("Exact duplicate rows removed:", before - len(df))

df["membership"] = df["membership"].astype("string").str.strip().str.title()
df["region"] = df["region"].astype("string").str.strip().str.title()

# Validate plausible ranges; mark invalid values as missing rather than silently deleting rows
df.loc[~df["age"].between(18, 100), "age"] = np.nan
df.loc[df["annual_income"] < 0, "annual_income"] = np.nan
df.loc[df["avg_order_value"] < 0, "avg_order_value"] = np.nan
df.loc[df["days_since_last_order"] < 0, "days_since_last_order"] = np.nan

print("Remaining missing values:")
display(df.isna().sum().to_frame("missing"))


## 6. Data transformation and Pandas grouping

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[17, 25, 35, 50, 65, 100],
                       labels=["18-25", "26-35", "36-50", "51-65", "66+"])
df["spending_band"] = pd.cut(df["avg_order_value"], bins=[0, 1000, 2500, 4500, np.inf],
                             labels=["Low", "Medium", "High", "Very High"])
df["engagement_score"] = df[["website_visits", "email_clicks"]].fillna(0).sum(axis=1)

display(df.sort_values("avg_order_value", ascending=False).head(10))
display(df.groupby("membership", observed=False).agg(
    customers=("customer_id", "count"),
    purchase_rate=("purchased", "mean"),
    mean_order_value=("avg_order_value", "mean")
).sort_values("purchase_rate", ascending=False))


## 7. NumPy numerical operations
At least three practical operations are shown: mean, percentile and conditional array calculation.

In [ ]:
income_array = df["annual_income"].dropna().to_numpy()
print("1. Mean annual income:", np.mean(income_array).round(2))
print("2. 25th, 50th, 75th percentiles:", np.percentile(income_array, [25, 50, 75]).round(2))
print("3. Customers with income above 60,000:", np.sum(income_array > 60000))
print("4. Clipped income array sample:", np.clip(income_array[:10], 20000, 100000))


## 8–9. Exploratory data analysis and visualisation
The charts below are generated from the practice dataset. Read each chart alongside its labels; the numeric patterns may change when a real dataset is substituted.

In [ ]:
# Chart 1: target distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="purchased")
plt.title("Purchase Outcome Distribution"); plt.xlabel("Purchased (0 = No, 1 = Yes)"); plt.ylabel("Customer records")
plt.tight_layout(); plt.show()

# Chart 2: age distribution
plt.figure(figsize=(7,4))
sns.histplot(data=df, x="age", bins=18, kde=True)
plt.title("Customer Age Distribution"); plt.xlabel("Age"); plt.ylabel("Count")
plt.tight_layout(); plt.show()

# Chart 3: average order value by purchase outcome
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x="purchased", y="avg_order_value")
plt.title("Average Order Value by Purchase Outcome"); plt.xlabel("Purchased (0/1)"); plt.ylabel("Average order value")
plt.tight_layout(); plt.show()

# Chart 4: purchase rate by membership
plt.figure(figsize=(7,4))
sns.barplot(data=df, x="membership", y="purchased", errorbar=None)
plt.title("Purchase Rate by Membership"); plt.xlabel("Membership"); plt.ylabel("Purchase rate")
plt.tight_layout(); plt.show()

# Chart 5: engagement versus purchase
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x="purchased", y="website_visits")
plt.title("Website Visits by Purchase Outcome"); plt.xlabel("Purchased (0/1)"); plt.ylabel("Website visits")
plt.tight_layout(); plt.show()

# Chart 6: numerical correlation heatmap
plt.figure(figsize=(9,6))
sns.heatmap(df.select_dtypes(include=np.number).drop(columns=["customer_id"]).corr(),
            annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Between Numerical Variables")
plt.tight_layout(); plt.show()

# Chart 7: purchase rate by age group
plt.figure(figsize=(8,4))
sns.barplot(data=df, x="age_group", y="purchased", errorbar=None)
plt.title("Purchase Rate by Age Group"); plt.xlabel("Age group"); plt.ylabel("Purchase rate")
plt.tight_layout(); plt.show()


### EDA insights to verify from generated charts
1. The target chart shows the balance between purchase and non-purchase outcomes; this matters when interpreting accuracy.
2. The age histogram describes the age coverage of the simulated customer base, not a real market.
3. The order-value boxplot compares transaction-value distributions across target groups.
4. Membership purchase rates indicate whether the simulated segments differ.
5. Website visits are an engagement indicator and can be compared between target groups.
6. The correlation heatmap helps identify linear relationships and possible redundancy, but correlation does not imply causation.
7. Age-group purchase rates help demonstrate segment-level comparison. Recalculate and rewrite these insights for the real client dataset before making business claims.

## 10. Statistical analysis and IQR outlier check

In [ ]:
key_vars = ["age", "annual_income", "avg_order_value", "website_visits", "previous_orders"]
display(df[key_vars].agg(["mean", "median", "std", "var", "min", "max"]).T)

outlier_summary = []
for col in key_vars:
    series = df[col].dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = ((series < lower) | (series > upper)).sum()
    outlier_summary.append({"variable": col, "Q1": q1, "Q3": q3, "IQR": iqr,
                            "lower_bound": lower, "upper_bound": upper, "potential_outliers": count})
outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)
print("Interpretation: IQR flags potential outliers, not necessarily errors. Investigate extreme values; retain valid customers unless there is a justified correction/removal rule.")


## 11–12. Feature engineering and ML problem definition
The target is binary `purchased`, so Logistic Regression is the primary model. `customer_id` is excluded because it is an identifier, not a meaningful predictor. Derived fields `age_group`, `spending_band` and `engagement_score` support exploration; to keep the first model simple and avoid redundant categorical/numeric features, use the original numeric predictors plus membership and region. The target is not included among predictors. In a real deployment, confirm every feature is available before the purchase decision to prevent leakage.

## 13–17. Train/test split, preprocessing, Logistic Regression and evaluation
A stratified 80/20 split preserves the target proportions. The test set is held out until evaluation. Missing numeric values are median-imputed and scaled; categorical values are mode-imputed and one-hot encoded. Preprocessing is fitted only on the training data through a pipeline to reduce leakage.

In [ ]:
features = ["age", "annual_income", "website_visits", "email_clicks",
            "previous_orders", "avg_order_value", "days_since_last_order",
            "membership", "region"]
X = df[features]
y = df["purchased"].astype(int)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}
display(pd.DataFrame(metrics.items(), columns=["Metric", "Score"]).round(3))
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted No", "Predicted Yes"],
            yticklabels=["Actual No", "Actual Yes"])
plt.title("Confusion Matrix"); plt.ylabel("Actual"); plt.xlabel("Predicted")
plt.tight_layout(); plt.show()

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Logistic Regression ROC Curve")
plt.tight_layout(); plt.show()

print("Test observations:", len(y_test))
print("Training observations:", len(y_train))


### Metric interpretation
- **Accuracy:** proportion of all test predictions that are correct.
- **Precision:** among predicted purchasers, the proportion who actually purchased.
- **Recall:** among actual purchasers, the proportion identified by the model.
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** how well the score ranks positive cases above negative cases across thresholds.
- **Confusion matrix:** counts correct and incorrect predictions by class.

No Linear Regression model is trained because the primary target is binary. Linear Regression is designed for continuous outcomes; a separate continuous target such as order value would be needed for a regression task. Metrics from this synthetic dataset are educational only and do not establish business readiness.

## 18. Client insights and recommendations
These are **hypotheses to test**, not validated claims about a real client:
1. Compare conversion rates by membership tier and test whether loyalty benefits improve incremental purchases.
2. Test targeted follow-up messages for customers with recent website visits but no purchase.
3. Review engagement channels such as email clicks and compare conversion by campaign.
4. Segment customers by prior order count and evaluate retention offers using controlled experiments.
5. Investigate high-value orders and extreme values before deciding whether they represent genuine premium customers or data errors.
6. Track model precision and recall together; choose the threshold based on the relative cost of missed buyers versus unnecessary outreach.
7. Monitor performance by region and age group to identify uneven model quality, while avoiding unfair or inappropriate targeting.
8. Retrain and re-evaluate periodically as purchasing behaviour changes.

### Limitations and next steps
- The data is simulated and may not reflect real customer behaviour.
- Correlation and predictive performance do not prove causality.
- Results depend on the generated sample and random seed.
- Before client use, load approved historical data, document consent/privacy safeguards, compare with a baseline, tune the threshold using validation data, and test on a later time period if predictions are intended for future customers.

## Conclusion
This notebook demonstrates a complete beginner workflow from data preparation through exploratory analysis and binary classification. Replace the synthetic dataset and refresh the insights, charts and metrics before submitting results as client-specific findings.